# 최소 실행 — 흐린 위성사진 선명하게

Sentinel-2가 찍은 인천 사진(10 m)을 3배 선명하게(3.33 m) 만든다. **셀 3개면 끝.**

- 구글 드라이브 **안 씀** (로그인·권한 승인 없음)
- `git clone` **안 함**
- `pip install` **안 함** (Colab에 이미 있는 것만 사용)
- 받는 것: 파일 2개, 합쳐서 **7 MB**
- GPU 없어도 됨

위에서부터 차례로 실행하세요.

## 1. 파일 2개 받기

In [ ]:
import urllib.request, os, time

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
FILES = {
    'weight.pt': f'{BASE}/models/01_edsr_x3/checkpoints/edsr_ikonosfull_x3_latest.pt',
    'input.png': f'{BASE}/dataset/test/incheon_600.png',
}

for name, url in FILES.items():
    if os.path.exists(name):
        print(f'{name} 이미 있음')
        continue
    t = time.time()
    urllib.request.urlretrieve(url, name)
    print(f'{name}  {os.path.getsize(name)/1e6:5.1f} MB  {time.time()-t:.1f}초')

import torch
print(f'\ntorch {torch.__version__} | 계산장치 {"GPU" if torch.cuda.is_available() else "CPU"}')

## 2. 학습에 쓴 데이터 보기

이 모델이 무엇을 보고 배웠는지 확인한다. 목록은 GitHub에서 읽고, 실제 사진은
**각 3쌍만** 내려받는다(약 1 MB).

- **연습문제(training)** — 선명한 정답을 3배 줄여 흐리게 만든 것
- **시험문제(validation)** — 같은 방식. 단 **학습에 안 쓴 지역**에서만

In [ ]:
import json, urllib.request
from collections import Counter

API = 'https://api.github.com/repos/BWMIN-Hub/SR_practice/contents/dataset'

def listing(split):
    with urllib.request.urlopen(f'{API}/{split}/HR') as r:
        return sorted(x['name'][:-4] for x in json.load(r))

def city(stem):
    return stem.rsplit('_y', 1)[0].rsplit('_', 1)[0]

sets = {}
for split in ['training', 'validation']:
    names = listing(split)
    sets[split] = names
    n = Counter(city(s) for s in names)
    print(f'{split:11s} {len(names):3d}쌍   ' + '  '.join(f'{k.replace("AOI_","")} {v}' for k, v in sorted(n.items())))

overlap = {city(s) for s in sets['training']} & {city(s) for s in sets['validation']}
print(f'\n같은 도시를 쓰지만 구역이 다르다: {", ".join(sorted(x.replace("AOI_","") for x in overlap))}')
print('시험문제 씬은 학습에 한 장도 쓰지 않았다.')

In [ ]:
import matplotlib.pyplot as plt
import imageio.v2 as imageio

def grab(split, stem):
    """HR/LR 한 쌍을 받아온다 (없으면 다운로드)."""
    out = []
    for sub, fn in [('HR', f'{stem}.png'), ('LR_bicubic/X3', f'{stem}x3.png')]:
        local = fn.replace('/', '_')
        if not os.path.exists(local):
            urllib.request.urlretrieve(f'{BASE}/dataset/{split}/{sub}/{fn}', local)
        out.append(imageio.imread(local))
    return out

N = 3
fig, ax = plt.subplots(4, N, figsize=(3.1 * N, 12.4))
for r, split in enumerate(['training', 'validation']):
    picks = sets[split][::max(1, len(sets[split]) // N)][:N]
    for c, stem in enumerate(picks):
        hr, lr = grab(split, stem)
        ax[2*r,   c].imshow(lr); ax[2*r,   c].set_title(f'{split} input {lr.shape[0]}px', fontsize=9)
        ax[2*r+1, c].imshow(hr); ax[2*r+1, c].set_title(f'{split} target {hr.shape[0]}px', fontsize=9)
        ax[2*r+1, c].set_xlabel(city(stem).replace('AOI_', ''), fontsize=8)
        for a in (ax[2*r, c], ax[2*r+1, c]):
            a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print('위 두 줄이 연습문제, 아래 두 줄이 시험문제. 각 쌍에서 위를 아래처럼 만드는 것이 과제다.')

## 3. 모델 만들고 실행

EDSR 구조를 여기에 직접 적어둔다. 따로 받아올 코드가 없다.

In [ ]:
import torch, torch.nn as nn, numpy as np, imageio.v2 as imageio

def conv(i, o, k=3):
    return nn.Conv2d(i, o, k, padding=k // 2)

class MeanShift(nn.Conv2d):
    """입력에서 평균색을 빼고 출력에서 도로 더한다 (학습 안정화용)."""
    def __init__(self, rgb_range, sign=-1):
        super().__init__(3, 3, 1)
        self.weight.data = torch.eye(3).view(3, 3, 1, 1)
        self.bias.data = sign * rgb_range * torch.tensor([0.4488, 0.4371, 0.4040])
        for p in self.parameters():
            p.requires_grad = False

class ResBlock(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.body = nn.Sequential(conv(n, n), nn.ReLU(True), conv(n, n))
    def forward(self, x):
        return self.body(x) + x

class EDSR(nn.Module):
    def __init__(self, n_resblocks=16, n_feats=64, scale=3):
        super().__init__()
        self.sub_mean, self.add_mean = MeanShift(255), MeanShift(255, 1)
        self.head = nn.Sequential(conv(3, n_feats))
        self.body = nn.Sequential(*[ResBlock(n_feats) for _ in range(n_resblocks)],
                                  conv(n_feats, n_feats))
        # 중첩 구조가 체크포인트 키(tail.0.0 / tail.1)와 맞아야 한다
        self.tail = nn.Sequential(
            nn.Sequential(conv(n_feats, n_feats * scale ** 2), nn.PixelShuffle(scale)),
            conv(n_feats, 3))
    def forward(self, x):
        x = self.head(self.sub_mean(x))
        return self.add_mean(self.tail(self.body(x) + x))

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net = EDSR().to(dev).eval()
net.load_state_dict(torch.load('weight.pt', map_location=dev))   # strict: 구조가 어긋나면 여기서 걸린다
print(f'모델 준비 완료 (파라미터 {sum(p.numel() for p in net.parameters())/1e6:.2f}M)')

lr = imageio.imread('input.png')
with torch.no_grad():
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(dev)
    sr = net(t).clamp(0, 255).round()[0].cpu().numpy().transpose(1, 2, 0).astype(np.uint8)

imageio.imwrite('output.png', sr)
print(f'{lr.shape[1]}x{lr.shape[0]}  ->  {sr.shape[1]}x{sr.shape[0]}   output.png 저장')

## 4. 결과 보기

In [ ]:
import cv2, matplotlib.pyplot as plt

bic = cv2.resize(lr, (sr.shape[1], sr.shape[0]), interpolation=cv2.INTER_CUBIC)

def sharp(a):
    return float(cv2.Laplacian(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), cv2.CV_64F).std())

print(f'Bicubic (just resize) : {sharp(bic):6.2f}')
print(f'EDSR    (this model)  : {sharp(sr):6.2f}')

g, S = cv2.cvtColor(sr, cv2.COLOR_RGB2GRAY), 300
best, bs = (0, 0), -1
for y in range(0, sr.shape[0] - S, S // 2):
    for x in range(0, sr.shape[1] - S, S // 2):
        v = g[y:y+S, x:x+S].std()
        if v > bs:
            best, bs = (y, x), v
y, x = best

fig, ax = plt.subplots(1, 3, figsize=(13, 4.6))
imgs = [cv2.resize(lr[y//3:y//3+S//3, x//3:x//3+S//3], (S, S), interpolation=cv2.INTER_NEAREST),
        bic[y:y+S, x:x+S], sr[y:y+S, x:x+S]]
for a, im, t in zip(ax, imgs, ['input (10 m)', 'Bicubic x3', 'EDSR x3']):
    a.imshow(im); a.set_title(t); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

---

가운데가 그냥 확대한 것, 오른쪽이 모델 결과다. 건물 경계와 도로가 더 또렷하면 성공.

`output.png` 는 왼쪽 폴더 아이콘에서 내려받을 수 있다.

지리정보가 붙은 GeoTIFF 출력, 큰 사진 처리, 학습까지 해보려면
[`01_edsr_x3.ipynb`](https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/01_edsr_x3.ipynb) 를 보세요.